# Phase 0 — Data hygiene & manifest


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


Build the manifest (checksums + row counts) from the raw CIC CSVs on Drive, then clean each dataset and write cleaned parquet to `interim/`.

In [ ]:
from pathlib import Path
from driftbench import config, manifest, clean_flows
import pandas as pd
config.ensure_dirs()

# 1) Manifest
m = manifest.build_manifest(config.RAW_DIR)
manifest.save_manifest(m, config.MANIFEST_DIR / 'dataset_manifest.json')
print('manifest written; verify:', manifest.verify_manifest(config.RAW_DIR, m))


In [ ]:
# 2) Clean each dataset. Concatenate that dataset's CSVs first.
for ds in config.DATASETS:
    csvs = sorted((config.RAW_DIR / ds).rglob('*.csv'))
    df = pd.concat([pd.read_csv(c, low_memory=False) for c in csvs], ignore_index=True)
    res = clean_flows(df)
    print(ds, '-> features', res.features.shape, 'dropped_const', len(res.dropped_constant),
          'dedup_removed', res.n_dedup_removed)
    out = config.INTERIM_DIR / ds
    out.mkdir(parents=True, exist_ok=True)
    res.features.to_parquet(out / 'features.parquet')
    res.metadata.to_parquet(out / 'metadata.parquet')
    res.labels.to_frame('label').to_parquet(out / 'labels.parquet')
